# hydra_pipeline

> Shared Hydra config helpers for `scripts/build.py` and `scripts/train.py`.

Channel presets, patch-cache orchestration, and stats loading — kept in the
library (not under `scripts/`) so `segpt-build` / `segpt-train` can import it
after `pip install` or `uv sync`.

In [ ]:
#| default_exp hydra_pipeline

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import json
from pathlib import Path

import cv2
from omegaconf import DictConfig

from segmentation_tools_pt.channels import CHANNEL_OPS, build_image, tophat
from segmentation_tools_pt.dataset import split_sample_ids, build_direct_dataloaders
from segmentation_tools_pt.dataset_build import build_patch_dataset
from segmentation_tools_pt.patches import PatchConfig
from segmentation_tools_pt.transforms import boundary_blur_kwargs_from_cfg, compute_dataset_stats

__all__ = [
    'CHANNEL_PRESETS',
    'resolve_channel_preset',
    'resolve_run_seed',
    'resolve_deterministic',
    'validate_patch_training_sizes',
    'patches_exist',
    'iter_train_images',
    'iter_raw_train_images',
    'load_or_compute_stats',
    'load_or_compute_stats_direct',
    'run_patch_pipeline',
    'run_direct_pipeline',
    'validate_data_mode_channels',
    'resolve_boundary_blur_cfg',
]

CHANNEL_OPS['ch2'] = lambda imgs: imgs[2]
CHANNEL_OPS['tophat_ch0'] = lambda imgs: tophat(imgs[0], 21)
CHANNEL_OPS['sobel_ch0'] = lambda imgs: cv2.convertScaleAbs(
    cv2.Sobel(imgs[0], cv2.CV_32F, 1, 1, ksize=3))

CHANNEL_PRESETS = {
    'single': dict(n_source_channels=1, channel_names=['ch0'], in_channels=1),
    'single_repeat_3': dict(
        n_source_channels=1, channel_names=['ch0', 'ch0', 'ch0'], in_channels=3),
    'single_to_3': dict(
        n_source_channels=1,
        channel_names=['ch0', 'tophat_ch0', 'sobel_ch0'],
        in_channels=3),
    'two_channel': dict(
        n_source_channels=2, channel_names=['ch0', 'ch1', 'product'], in_channels=3),
    'three_channel': dict(
        n_source_channels=3, channel_names=['ch0', 'ch1', 'ch2'], in_channels=3),
}


In [ ]:
#| export
def resolve_channel_preset(data_mode: str) -> dict:
    """Map `data.data_mode` to source channel count, names, and model `in_channels`.

    Example:
        >>> resolve_channel_preset('single')['in_channels']
        1
    """
    if data_mode not in CHANNEL_PRESETS:
        raise ValueError(
            f'Unknown data.data_mode={data_mode!r}. '
            f'Choose one of: {list(CHANNEL_PRESETS)}'
        )
    return CHANNEL_PRESETS[data_mode]


def resolve_run_seed(cfg: DictConfig) -> int:
    """Return the run seed, preferring `train.seed` and falling back to `patch.seed`."""
    return int(cfg.train.get('seed', cfg.patch.seed))


def resolve_deterministic(cfg: DictConfig) -> bool:
    """Return whether the run should request deterministic torch backends."""
    return bool(cfg.train.get('deterministic', True))


def validate_patch_training_sizes(cfg: DictConfig) -> None:
    """Fail early when the model crop is larger than the saved patch size."""
    patch_size = int(cfg.patch.patch_size)
    crop_size = int(cfg.train.crop_size)
    if crop_size > patch_size:
        raise ValueError(
            'Invalid patch-mode configuration: '
            f'train.crop_size={crop_size} is larger than patch.patch_size={patch_size}. '
            'In patch mode the model crop must fit inside the saved patch. '
            'Use a larger patch.patch_size, or reduce train.crop_size.'
        )

In [ ]:
# test
assert resolve_channel_preset('single_repeat_3')['in_channels'] == 3

In [ ]:
#| export
def resolve_boundary_blur_cfg(cfg: DictConfig) -> dict:
    """Map Hydra `augment.boundary_*` keys to `get_train_transforms(boundary_blur=...)`."""
    return boundary_blur_kwargs_from_cfg(cfg.augment)


In [ ]:
#| export
def patches_exist(patch_dir: Path) -> bool:
    """Return True when `patch_dir/metadata.json` lists train patches."""
    meta = patch_dir / 'metadata.json'
    if not meta.exists():
        return False
    data = json.loads(meta.read_text())
    return bool(data.get('train'))


def iter_train_images(
    patch_dir: Path,
    n_source_channels: int,
    channel_names: list[str],
    split: str = 'train',
):
    """Yield assembled (H, W, C) model inputs from saved patch source channels."""
    img_dir = patch_dir / split / 'images'
    meta = json.loads((patch_dir / 'metadata.json').read_text())
    for m in meta[split]:
        sid = m['sample_id']
        chans = []
        for i in range(n_source_channels):
            paths = [img_dir / f'{sid}_ch{i}.png']
            if i == 0 and n_source_channels == 1:
                paths.insert(0, img_dir / f'{sid}.png')
            arr = None
            for p in paths:
                arr = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                if arr is not None:
                    break
            if arr is None:
                break
            chans.append(arr)
        if len(chans) != n_source_channels:
            continue
        yield build_image(chans, channel_names)

In [ ]:
#| export
def load_or_compute_stats(
    patch_dir: Path,
    n_source_channels: int,
    channel_names: list[str],
) -> tuple[tuple[float, ...], tuple[float, ...]]:
    """Load or compute per-channel mean/std from training patches."""
    stats_path = patch_dir / 'stats.json'
    if stats_path.exists():
        cached = json.loads(stats_path.read_text())
        mean = tuple(cached['mean'])
        std = tuple(cached['std'])
        print(f'Loaded stats from {stats_path}')
        return mean, std

    print('Computing per-channel mean/std from training patches …')
    mean, std = compute_dataset_stats(
        iter_train_images(patch_dir, n_source_channels, channel_names)
    )
    stats_path.write_text(json.dumps({'mean': list(mean), 'std': list(std)}, indent=2))
    print(f'Saved stats → {stats_path}')
    return mean, std


def load_or_compute_stats_direct(
    raw_dir: Path,
    train_ids: list[str],
    n_source_channels: int,
    channel_names: list[str],
    exts: str,
) -> tuple[tuple[float, ...], tuple[float, ...]]:
    """Load or compute per-channel mean/std from raw training images."""
    stats_path = raw_dir / 'stats.json'
    if stats_path.exists():
        cached = json.loads(stats_path.read_text())
        mean = tuple(cached['mean'])
        std = tuple(cached['std'])
        print(f'Loaded stats from {stats_path}')
        return mean, std

    print('Computing per-channel mean/std from raw training images …')
    mean, std = compute_dataset_stats(
        iter_raw_train_images(raw_dir, train_ids, n_source_channels, channel_names, exts)
    )
    stats_path.write_text(json.dumps({'mean': list(mean), 'std': list(std)}, indent=2))
    print(f'Saved stats → {stats_path}')
    return mean, std

In [ ]:
#| export
def run_patch_pipeline(cfg, raw_dir, patch_dir, n_source, channel_names, mask_suffix):
    """Build or reuse patch cache; return `(mean, std, patch_dir_str)`."""
    patch_cfg = PatchConfig(
        patch_size=cfg.patch.patch_size,
        stride=cfg.patch.stride,
        min_fg_pixels=cfg.patch.min_fg_pixels,
        max_empty_fraction=cfg.patch.max_empty_frac,
        val_fraction=cfg.patch.val_fraction,
        seed=cfg.patch.seed,
    )

    if cfg.data.force_rebuild or not patches_exist(patch_dir):
        print(f'\nBuilding patch dataset → {patch_dir}')
        patch_dir.mkdir(parents=True, exist_ok=True)
        meta = build_patch_dataset(
            source_dir=str(raw_dir),
            output_dir=str(patch_dir),
            cfg=patch_cfg,
            num_classes=cfg.train.num_classes,
            n_channels=n_source,
            n_jobs=cfg.patch.num_workers,
            fit_rects=cfg.data.fit_rects,
            image_exts=cfg.data.exts,
            mask_suffix=mask_suffix,
        )
        print(
            f'  train patches: {len(meta["train"])}  |  '
            f'val patches: {len(meta["val"])}'
        )
    else:
        print(f'\nReusing existing patches in {patch_dir} (set data.force_rebuild=true to rebuild)')

    mean, std = load_or_compute_stats(patch_dir, n_source, channel_names)
    return mean, std, str(patch_dir)


def run_direct_pipeline(cfg, raw_dir, sample_ids, n_source, channel_names, mask_suffix):
    """Split raw samples and compute stats; return `(mean, std, train_loader, val_loader)`."""
    train_ids, val_ids = split_sample_ids(
        sample_ids, cfg.patch.val_fraction, cfg.patch.seed,
    )
    print(f'\nDirect mode: {len(train_ids)} train / {len(val_ids)} val full images')
    mean, std = load_or_compute_stats_direct(
        raw_dir, train_ids, n_source, channel_names, cfg.data.exts,
    )
    boundary_blur = resolve_boundary_blur_cfg(cfg)
    train_loader, val_loader, _, _ = build_direct_dataloaders(
        raw_dir=str(raw_dir),
        train_ids=train_ids,
        val_ids=val_ids,
        batch_size=cfg.train.batch_size,
        num_workers=cfg.train.num_workers,
        crop_size=cfg.train.crop_size,
        num_classes=cfg.train.num_classes,
        n_source_channels=n_source,
        channel_names=channel_names,
        mask_suffix=mask_suffix,
        exts=cfg.data.exts,
        fit_rects=cfg.data.fit_rects,
        mean=mean,
        std=std,
        boundary_blur=boundary_blur,
        flips=bool(cfg.augment.get('flips', False)),
        seed=resolve_run_seed(cfg),
    )
    return mean, std, train_loader, val_loader


def validate_data_mode_channels(cfg: DictConfig) -> tuple[int, int, list[str]]:
    """Resolve channel preset and validate `model.in_channels` against `data_mode`."""
    preset = resolve_channel_preset(cfg.data.data_mode)
    n_source = preset['n_source_channels']
    channel_names = preset['channel_names']
    in_channels = int(cfg.model.in_channels)
    expected = preset['in_channels']
    if in_channels != expected:
        raise ValueError(
            f'model.in_channels={in_channels} does not match '
            f'data.data_mode={cfg.data.data_mode!r} '
            f'(expected {expected}). '
            f'Update model.in_channels in your config to match data_mode.'
        )
    return n_source, in_channels, channel_names
